# Tầng 2 của hệ thống kiểm duyệt: PhoBERT-CNN cải tiến và các mô hình so sánh

`BACKBONE` chọn mô hình: `phobert_cnn` (mặc định), `phobert_cls`, `visobert_cls`, `visobert_cnn`. ViSoBERT đọc văn bản gốc nên bỏ qua bước tách từ VnCoreNLP. Mọi mô hình dùng chung dữ liệu, seed, cách chọn checkpoint và các phép đo.

Notebook chạy **một giai đoạn thí nghiệm mỗi lần** (đổi `STAGE` rồi Run All). Mỗi giai đoạn chỉ thêm một thay đổi so với giai đoạn trước, để bảng kết quả cho biết thay đổi nào có tác dụng.

| `STAGE` | Thay đổi | Mục đích |
|---|---|---|
| `0_baseline` | PhoBERT-CNN gốc trên dữ liệu đã làm sạch | Mốc so sánh |
| `1_recipe` | + dừng sớm, label smoothing 0,1, LLRD 0,9, R-Drop | Giảm quá khớp / quá tự tin |
| `2_quote` | + nối cặp text – quote (`text_quote`) | Ngữ cảnh quote có giúp không |
| `3_relabel` | + nhãn đã được nhóm sửa (`label_fixes.csv`) | Giảm nhiễu nhãn |

Mỗi giai đoạn chạy 3 seed và báo cáo mean ± std của: Macro-F1, F1 từng lớp, F1 nhóm có/không quote, **ECE** (trước và sau temperature scaling) và **coverage** của tầng 2 ở độ chính xác mục tiêu (bao nhiêu bình luận tầng 2 tự quyết, không phải đẩy lên Qwen).

Dữ liệu đầu vào: các file `*_clean.csv` từ `clean_data.py` (đã bỏ chữ ký thiết bị và tiền tố `said:`), nên notebook không làm lại bước đó.

In [ ]:
%pip install -q transformers accelerate py_vncorenlp sentencepiece

## 0. Cấu hình

In [ ]:
import os, json, random, time, gc, hashlib, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.optimize import minimize_scalar

# ===== Đường dẫn: tải du_lieu_da_lam_sach.zip lên Kaggle thành một dataset rồi sửa DATA_DIR =====
DATA_DIR = "/kaggle/input/voz-toxic-clean"
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.path.abspath("."))   # tự nhận Kaggle / Colab / máy cục bộ
LABEL_FIX_PATH = os.path.join(DATA_DIR, "label_fixes.csv")   # chỉ dùng ở STAGE 3

# ===== Chọn giai đoạn =====
STAGE = "1_recipe"            # "0_baseline" | "1_recipe" | "2_quote" | "3_relabel"
STAGE3_INPUT_MODE = "text_only"   # điền chế độ tốt hơn sau khi so sánh giai đoạn 1 và 2

PRESETS = {
    "0_baseline": dict(input_mode="text_only", label_smoothing=0.0, llrd=1.0, rdrop_alpha=0.0, patience=None, use_fixes=False),
    "1_recipe":   dict(input_mode="text_only", label_smoothing=0.1, llrd=0.9, rdrop_alpha=1.0, patience=2,    use_fixes=False),
    "2_quote":    dict(input_mode="text_quote", label_smoothing=0.1, llrd=0.9, rdrop_alpha=1.0, patience=2,   use_fixes=False),
    "3_relabel":  dict(input_mode=STAGE3_INPUT_MODE, label_smoothing=0.1, llrd=0.9, rdrop_alpha=1.0, patience=2, use_fixes=True),
}
OVERRIDES = {}                # ví dụ {"rdrop_alpha": 0.0} để làm ablation một thành phần
CFG = {**PRESETS[STAGE], **OVERRIDES}
INPUT_MODE = CFG["input_mode"]

SEEDS = [42, 13, 2024]

# ===== Mô hình =====
BACKBONE = "phobert_cnn"      # "phobert_cnn" | "phobert_cls" | "visobert_cls" | "visobert_cnn"
BACKBONES = {   # tên Hugging Face, đầu phân loại, có cần tách từ VnCoreNLP không
    "phobert_cnn":  ("vinai/phobert-base", "cnn", True),
    "phobert_cls":  ("vinai/phobert-base", "cls", True),
    "visobert_cls": ("uitnlp/visobert",    "cls", False),
    "visobert_cnn": ("uitnlp/visobert",    "cnn", False),
}
MODEL_NAME, HEAD, NEEDS_SEG = BACKBONES[BACKBONE]
DYNAMIC_PADDING = True        # pad theo câu dài nhất trong batch thay vì luôn pad tới MAX_LEN (nhanh hơn nhiều, kết quả tương đương)
MEASURE_CPU = True            # đo thêm độ trễ trên CPU ở mục 9 (mất vài phút)
MAX_LEN = 256 if INPUT_MODE == "text_quote" else 160
BATCH_SIZE = 16               # CUDA OOM -> 8
EPOCHS = 5
ENCODER_LR = 2e-5             # lr của lớp encoder trên cùng; các lớp dưới nhân thêm llrd^k
HEAD_LR = 1e-3 if HEAD == "cnn" else 1e-4   # CNN khởi tạo ngẫu nhiên cần lr lớn; đầu [CLS] chỉ là 1 lớp linear
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
KERNEL_SIZES = (2, 3, 4)
NUM_FILTERS = 128
DROPOUT = 0.30
USE_LAST_K_LAYERS = 4
CLASS_WEIGHT = None           # None | "balanced"
TARGET_ACC = 0.97             # độ chính xác mong muốn trên phần tầng 2 tự quyết (để tính coverage)

TAG = f"{BACKBONE}_{STAGE}" + ("_" + "_".join(f"{k}{v}" for k, v in OVERRIDES.items()) if OVERRIDES else "")
LABELS = ["clean", "offensive", "hate"]
LABEL_TO_ID = {l: i for i, l in enumerate(LABELS)}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT_DIR, exist_ok=True)
print("BACKBONE:", BACKBONE, MODEL_NAME, "| head:", HEAD, "| tách từ:", NEEDS_SEG); print("STAGE:", STAGE, "| TAG:", TAG); print(json.dumps(CFG, indent=2))
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## 1. Đọc dữ liệu đã làm sạch (và áp nhãn đã sửa nếu là giai đoạn 3)

In [ ]:
def read_split(name):
    d = pd.read_csv(os.path.join(DATA_DIR, f"{name}_clean.csv"))
    d["row_id"] = np.arange(len(d))          # khớp với row_id trong label_fixes.csv
    d["text"] = d["text"].fillna("").astype(str)
    d["quoted_text"] = d["quoted_text"].fillna("").astype(str)
    d["label_str"] = d["label"].astype(str).str.strip().str.lower()
    bad = ~d["label_str"].isin(LABELS)
    if bad.any(): raise ValueError(f"{name}: nhãn lạ {d.loc[bad, 'label_str'].unique()[:5]}")
    d["label"] = d["label_str"].map(LABEL_TO_ID).astype(int)
    return d.drop(columns=[c for c in ["text_raw", "quoted_raw"] if c in d.columns])

train_df, val_df, test_df = read_split("train"), read_split("val"), read_split("test")

if CFG["use_fixes"] and not os.path.exists(LABEL_FIX_PATH):
    print(f"Không thấy {LABEL_FIX_PATH}: coi như nhãn đã được sửa sẵn trong train/val, bỏ qua bước áp nhãn sửa.")
elif CFG["use_fixes"]:
    fixes = pd.read_csv(LABEL_FIX_PATH).drop_duplicates(["split", "row_id"], keep="last")
    if (fixes["split"] == "test").any(): raise ValueError("Không sửa nhãn tập test bằng file này – test phải giữ nguyên để so sánh công bằng.")
    for name, d in [("train", train_df), ("val", val_df)]:
        f = fixes[fixes["split"] == name]
        idx = d.index[d["row_id"].isin(f["row_id"])]
        m = f.set_index("row_id")
        rid = d.loc[idx, "row_id"]
        if len(idx) and (d.loc[idx, "label_str"].values == m.loc[rid, "old_label"].values).mean() < 1.0:
            if (d.loc[idx, "label_str"].values == m.loc[rid, "new_label"].values).mean() == 1.0:
                print(f"{name}: các nhãn này đã được sửa sẵn trong file dữ liệu, bỏ qua"); continue
            raise ValueError(f"{name}: old_label trong label_fixes.csv không khớp dữ liệu – có thể sai phiên bản file")
        d.loc[idx, "label"] = m.loc[rid, "new_label"].map(LABEL_TO_ID).values
        print(f"{name}: đã sửa {len(idx)} nhãn")

for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    vc = d["label"].value_counts().reindex([0, 1, 2]).fillna(0).astype(int)
    print(f"{n:5s} n={len(d):6d} | clean={vc[0]} offensive={vc[1]} hate={vc[2]} | có quote={(d['quoted_text'].str.strip() != '').mean():.1%}")
fp = {n: hashlib.md5("\n".join(d["text"] + "||" + d["label"].astype(str)).encode()).hexdigest()[:12]
      for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
print("Fingerprint (test phải giống nhau giữa mọi giai đoạn):", fp)

## 2. Tách từ VnCoreNLP (chỉ cho PhoBERT; cache dùng chung cho mọi giai đoạn)

In [ ]:
def identity(x): return x
if NEEDS_SEG:
    import py_vncorenlp
    VNCORE_DIR = os.path.abspath(os.path.join(OUT_DIR, "vncorenlp"))
    os.makedirs(VNCORE_DIR, exist_ok=True)
    _cwd = os.getcwd()
    if not os.path.exists(os.path.join(VNCORE_DIR, "VnCoreNLP-1.2.jar")):
        py_vncorenlp.download_model(save_dir=VNCORE_DIR)
    segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=VNCORE_DIR)
    os.chdir(_cwd)   # py_vncorenlp đổi thư mục làm việc; trả lại để đường dẫn tương đối không hỏng

    SEG_CACHE = os.path.join(OUT_DIR, "seg_cache.csv")
    cache = {}
    if os.path.exists(SEG_CACHE):
        c = pd.read_csv(SEG_CACHE, keep_default_na=False, dtype=str)
        cache = dict(zip(c["raw"], c["seg"]))
    need = sorted({s for d in (train_df, val_df, test_df) for col in ("text", "quoted_text") for s in d[col] if s.strip()} - set(cache))
    print(f"Cache có {len(cache)} chuỗi, cần tách thêm {len(need)}")
    t0 = time.time()
    for i, s in enumerate(need, 1):
        cache[s] = " ".join(segmenter.word_segment(s))
        if i % 5000 == 0: print(f"  {i}/{len(need)} ({(time.time()-t0)/60:.1f} phút)")
    if need:
        pd.DataFrame({"raw": list(cache), "seg": list(cache.values())}).to_csv(SEG_CACHE, index=False)
    for d in (train_df, val_df, test_df):
        d["text_seg"] = d["text"].map(lambda s: cache.get(s, ""))
        d["quote_seg"] = d["quoted_text"].map(lambda s: cache.get(s, ""))
    print(train_df[["text", "text_seg"]].head(3).to_string())

    segment_fn = lambda s: " ".join(segmenter.word_segment(s)) if s.strip() else ""
else:
    segmenter, segment_fn = None, identity
    for d in (train_df, val_df, test_df):
        d["text_seg"], d["quote_seg"] = d["text"], d["quoted_text"]
    print(f"{BACKBONE}: dùng văn bản gốc, không tách từ")

## 3. Dataset

In [ ]:
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def encode(t, q=""):
    """Trả về list token id (chưa pad). text_quote: cắt quote trước để giữ nguyên bình luận."""
    if INPUT_MODE == "text_quote" and q.strip():
        trunc = "only_second" if len(tokenizer.tokenize(t)) < MAX_LEN - 40 else "longest_first"
        return tokenizer(t, q, max_length=MAX_LEN, truncation=trunc)["input_ids"]
    return tokenizer(t, max_length=MAX_LEN, truncation=True)["input_ids"]

PAD_ID = tokenizer.pad_token_id
def collate(batch, dynamic=None):
    dynamic = DYNAMIC_PADDING if dynamic is None else dynamic
    L = max(len(b["ids"]) for b in batch) if dynamic else MAX_LEN
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long); am = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        ids[i, :len(b["ids"])] = torch.tensor(b["ids"]); am[i, :len(b["ids"])] = 1
    return {"input_ids": ids, "attention_mask": am, "labels": torch.tensor([b["y"] for b in batch])}

class CommentDataset(Dataset):
    def __init__(self, frame):     # tokenize một lần lúc tạo dataset
        self.ids = [encode(t, q) for t, q in zip(frame["text_seg"], frame["quote_seg"])]
        self.y = frame["label"].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return {"ids": self.ids[i], "y": self.y[i]}

t0 = time.time()
train_ds, val_ds, test_ds = CommentDataset(train_df), CommentDataset(val_df), CommentDataset(test_df)
print(f"Tokenize xong trong {time.time()-t0:.0f}s | độ dài token trung vị (train): {int(np.median([len(x) for x in train_ds.ids]))}, "
      f"p95: {int(np.percentile([len(x) for x in train_ds.ids], 95))}, bị cắt ở MAX_LEN: {np.mean([len(x) >= MAX_LEN for x in train_ds.ids]):.1%}")
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate)
y_val, y_test = val_df["label"].values, test_df["label"].values
has_q_test = (test_df["quoted_text"].str.strip() != "").values
print("Train/Val/Test:", len(train_ds), len(val_ds), len(test_ds), "| MAX_LEN:", MAX_LEN)

## 4. Mô hình: bộ mã hoá (PhoBERT / ViSoBERT) + đầu CNN hoặc [CLS]
Tên các lớp giữ như notebook cũ (`encoder`, `convs`, `drop`, `fc`) để checkpoint PhoBERT-CNN cũ vẫn nạp được.

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, model_name, head="cnn", num_labels=3, kernel_sizes=(2, 3, 4), num_filters=128, dropout=0.3, last_k=4):
        super().__init__()
        self.head, self.last_k = head, last_k
        self.encoder = AutoModel.from_pretrained(model_name, output_hidden_states=(head == "cnn"))
        H = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        if head == "cnn":
            self.convs = nn.ModuleList([nn.Conv1d(H * last_k, num_filters, k, padding=k // 2) for k in kernel_sizes])
            self.fc = nn.Linear(num_filters * len(kernel_sizes), num_labels)
        else:
            self.fc = nn.Linear(H, num_labels)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if self.head == "cls":
            return self.fc(self.drop(out.last_hidden_state[:, 0]))
        x = torch.cat(out.hidden_states[-self.last_k:], dim=-1)
        x = (x * attention_mask.unsqueeze(-1).to(x.dtype)).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            c = torch.relu(conv(x))[:, :, : x.size(2)]
            c = c.masked_fill(attention_mask.unsqueeze(1) == 0, -1e4)
            pooled.append(c.max(dim=2).values)
        return self.fc(self.drop(torch.cat(pooled, dim=1)))

def build_model():
    return TextClassifier(MODEL_NAME, HEAD, 3, KERNEL_SIZES, NUM_FILTERS, DROPOUT, USE_LAST_K_LAYERS)

## 5. Huấn luyện

- **LLRD** (layer-wise learning-rate decay): lớp encoder thứ *i* (đếm từ trên xuống) có lr = `ENCODER_LR × llrd^i`, embedding thấp nhất. Lớp dưới giữ kiến thức chung, lớp trên thích nghi nhiều hơn. `llrd = 1.0` là tắt.
- **Label smoothing**: nhãn đích 0,9/0,05/0,05 thay vì 1/0/0, giảm việc mô hình đẩy xác suất về 1 trên các nhãn có thể bị nhiễu.
- **R-Drop**: mỗi batch chạy hai lần với dropout khác nhau và phạt độ lệch KL giữa hai dự đoán. Tốn khoảng 1,7–2× thời gian mỗi epoch.
- **Dừng sớm** theo val Macro-F1 (`patience` epoch không cải thiện). Checkpoint tốt nhất giữ trong RAM.

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def build_optimizer(m):
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    layers = m.encoder.encoder.layer; L = len(layers); groups = []
    def add(params, lr):
        p = list(params)
        groups.append({"params": [x for n, x in p if not n.endswith(no_decay)], "lr": lr, "weight_decay": WEIGHT_DECAY})
        groups.append({"params": [x for n, x in p if n.endswith(no_decay)], "lr": lr, "weight_decay": 0.0})
    for i, layer in enumerate(layers):                       # i = 0 là lớp dưới cùng
        add(layer.named_parameters(), ENCODER_LR * CFG["llrd"] ** (L - 1 - i))
    add(m.encoder.embeddings.named_parameters(), ENCODER_LR * CFG["llrd"] ** L)
    add([(n, p) for n, p in m.named_parameters() if not n.startswith("encoder.")], HEAD_LR)
    return torch.optim.AdamW([g for g in groups if g["params"]])

cw = None
if CLASS_WEIGHT == "balanced":
    cnt = np.bincount(train_df["label"], minlength=3); cw = torch.tensor(len(train_df) / (3 * cnt), dtype=torch.float, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=CFG["label_smoothing"])
use_amp = DEVICE.type == "cuda"
def mv(b): return {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}

def train_step_loss(m, b):
    logits = m(b["input_ids"], b["attention_mask"]).float()
    loss = criterion(logits, b["labels"])
    a = CFG["rdrop_alpha"]
    if a > 0:
        logits2 = m(b["input_ids"], b["attention_mask"]).float()
        loss = 0.5 * (loss + criterion(logits2, b["labels"]))
        lp, lq = F.log_softmax(logits, -1), F.log_softmax(logits2, -1)
        kl = 0.5 * (F.kl_div(lp, lq, log_target=True, reduction="batchmean") + F.kl_div(lq, lp, log_target=True, reduction="batchmean"))
        loss = loss + a * kl
    return loss

@torch.no_grad()
def predict_logits(m, loader):
    m.eval(); out = []
    for b in loader:
        b = mv(b)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            out.append(m(b["input_ids"], b["attention_mask"]).float().cpu())
    return torch.cat(out).numpy()

def macro_f1(y, p): return f1_score(y, p, labels=[0, 1, 2], average="macro", zero_division=0)

def train_one_seed(seed):
    set_seed(seed)
    gen = torch.Generator(); gen.manual_seed(seed)
    tr_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=gen, pin_memory=use_amp, collate_fn=collate)
    m = build_model().to(DEVICE)
    opt = build_optimizer(m)
    if use_amp: torch.cuda.reset_peak_memory_stats()
    steps = len(tr_dl) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best = {"f1": -1, "epoch": 0, "state": None}; hist = []; bad = 0
    for ep in range(1, EPOCHS + 1):
        m.train(); t0 = time.time(); run = 0.0; seen = 0
        for step, b in enumerate(tr_dl, 1):
            b = mv(b); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                loss = train_step_loss(m, b)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            run += loss.item() * len(b["labels"]); seen += len(b["labels"])
            if step % 200 == 0: print(f"  seed {seed} ep {ep} step {step}/{len(tr_dl)} loss={run/seen:.4f}", flush=True)
        if use_amp: torch.cuda.synchronize()
        train_sec = time.time() - t0
        lv = predict_logits(m, val_dl)
        vf1 = macro_f1(y_val, lv.argmax(1))
        vloss = float(F.cross_entropy(torch.tensor(lv), torch.tensor(y_val)).item())   # NLL thuần, không smoothing
        hist.append({"seed": seed, "epoch": ep, "train_loss": run / seen, "val_nll": vloss, "val_macro_f1": vf1, "minutes": (time.time() - t0) / 60,
                     "train_samples_per_s": seen / train_sec,
                     "peak_vram_train_mb": torch.cuda.max_memory_allocated() / 2**20 if use_amp else None})
        print(f"[seed {seed} | epoch {ep}] train_loss={run/seen:.4f} val_nll={vloss:.4f} val_macro_f1={vf1:.4f} ({hist[-1]['minutes']:.1f} phút)", flush=True)
        if vf1 > best["f1"]:
            best = {"f1": vf1, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}}; bad = 0
        else:
            bad += 1
            if CFG["patience"] is not None and bad >= CFG["patience"]:
                print(f"  Dừng sớm ở epoch {ep}"); break
    m.load_state_dict(best["state"])
    best["train_samples_per_s"] = float(np.mean([h_["train_samples_per_s"] for h_ in hist]))
    best["peak_vram_train_mb"] = max((h_["peak_vram_train_mb"] or 0) for h_ in hist) if use_amp else None
    best["epochs_run"] = len(hist)
    return m, best, hist

## 6. Hiệu chỉnh và các chỉ số cho cascade

In [ ]:
def softmax(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def fit_temperature(logits, y):
    """Chọn T > 0 cực tiểu NLL trên tập val (Guo et al. 2017)."""
    def nll(logT):
        p = softmax(logits / np.exp(logT)); return -np.log(p[np.arange(len(y)), y] + 1e-12).mean()
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

def ece(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def nll(probs, y): return float(-np.log(probs[np.arange(len(y)), y] + 1e-12).mean())

def coverage_at_acc(p_val, yv, p_test, yt, target=TARGET_ACC):
    """Chọn τ nhỏ nhất trên VAL sao cho độ chính xác phần tự quyết >= target, rồi đo trên TEST."""
    cv, pv = p_val.max(1), p_val.argmax(1); tau = None
    for t in np.round(np.arange(0.50, 0.996, 0.005), 3):
        m = cv >= t
        if m.sum() >= 20 and (pv[m] == yv[m]).mean() >= target: tau = float(t); break
    if tau is None: return {"tau": None, "coverage": 0.0, "acc_decided": float("nan")}
    ct, pt = p_test.max(1), p_test.argmax(1); m = ct >= tau
    return {"tau": tau, "coverage": float(m.mean()), "acc_decided": float((pt[m] == yt[m]).mean()) if m.any() else float("nan")}

def all_metrics(seed, best, lv, lt):
    T = fit_temperature(lv, y_val)
    p_raw, p_cal, pv_cal = softmax(lt), softmax(lt / T), softmax(lv / T)
    pred = lt.argmax(1); pc = f1_score(y_test, pred, labels=[0, 1, 2], average=None, zero_division=0)
    cov = coverage_at_acc(pv_cal, y_val, p_cal, y_test)
    return {"stage": STAGE, "tag": TAG, "seed": seed, "best_epoch": best["epoch"], "val_macro_f1": best["f1"],
            "test_macro_f1": macro_f1(y_test, pred), "test_acc": accuracy_score(y_test, pred),
            "f1_clean": pc[0], "f1_offensive": pc[1], "f1_hate": pc[2],
            "macro_f1_has_quote": macro_f1(y_test[has_q_test], pred[has_q_test]),
            "macro_f1_no_quote": macro_f1(y_test[~has_q_test], pred[~has_q_test]),
            "temperature": T, "ece_raw": ece(p_raw, y_test), "ece_cal": ece(p_cal, y_test),
            "nll_raw": nll(p_raw, y_test), "nll_cal": nll(p_cal, y_test),
            "tau2": cov["tau"], "coverage_tier2": cov["coverage"], "acc_decided_tier2": cov["acc_decided"]}

## 7. Chạy các seed

In [ ]:
RES_PATH = os.path.join(OUT_DIR, f"results_{TAG}.csv")
HIST_PATH = os.path.join(OUT_DIR, f"history_{TAG}.csv")
results, histories, best_overall = [], [], {"val": -1}
for seed in SEEDS:
    t0 = time.time()
    m, best, hist = train_one_seed(seed)
    lv, lt = predict_logits(m, val_dl), predict_logits(m, test_dl)
    np.savez_compressed(os.path.join(OUT_DIR, f"logits_{TAG}_seed{seed}.npz"), val=lv, test=lt, y_val=y_val, y_test=y_test)
    r = all_metrics(seed, best, lv, lt); r["train_minutes"] = (time.time() - t0) / 60
    r.update({"backbone": BACKBONE, "epochs_run": best["epochs_run"], "train_samples_per_s": best["train_samples_per_s"],
              "peak_vram_train_mb": best["peak_vram_train_mb"]})
    results.append(r); histories += hist
    pd.DataFrame(results).to_csv(RES_PATH, index=False); pd.DataFrame(histories).to_csv(HIST_PATH, index=False)
    print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}, ensure_ascii=False))
    if best["f1"] > best_overall["val"]:   # giữ checkpoint của seed có val tốt nhất cho tầng 2
        best_overall = {"val": best["f1"], "seed": seed, "T": r["temperature"], "tau": r["tau2"]}
        torch.save({"model_state_dict": best["state"], "model_name": MODEL_NAME, "max_len": MAX_LEN,
                    "kernel_sizes": list(KERNEL_SIZES), "num_filters": NUM_FILTERS, "dropout": DROPOUT,
                    "last_k": USE_LAST_K_LAYERS, "input_mode": INPUT_MODE, "stage": STAGE, "seed": seed,
                    "val_macro_f1": best["f1"], "label_names": LABELS,
                    "backbone": BACKBONE, "head": HEAD, "needs_seg": NEEDS_SEG},
                   os.path.join(OUT_DIR, f"{TAG}_best.pt"))
        json.dump({"temperature": r["temperature"], "threshold": r["tau2"] if r["tau2"] is not None else 0.99},
                  open(os.path.join(OUT_DIR, f"{TAG}_calibration.json"), "w"), indent=2)
    del m; gc.collect(); torch.cuda.empty_cache()
print("Checkpoint tầng 2:", f"{TAG}_best.pt", "| seed", best_overall["seed"])

## 8. Tổng kết mean ± std

In [ ]:
res = pd.DataFrame(results)
cols = ["val_macro_f1", "test_macro_f1", "test_acc", "f1_clean", "f1_offensive", "f1_hate", "macro_f1_has_quote",
        "macro_f1_no_quote", "ece_raw", "ece_cal", "nll_raw", "nll_cal", "temperature", "coverage_tier2", "acc_decided_tier2", "best_epoch", "epochs_run", "train_minutes",
        "train_samples_per_s", "peak_vram_train_mb"]
summary = pd.DataFrame({"mean": res[cols].astype(float).mean(), "std": res[cols].astype(float).std(ddof=1)})
summary["mean ± std"] = [f"{a:.4f} ± {b:.4f}" for a, b in zip(summary["mean"], summary["std"])]
display(summary[["mean ± std"]])
json.dump({"stage": STAGE, "tag": TAG, "cfg": CFG, "n_seeds": len(res),
           **{c: {"mean": float(summary.loc[c, "mean"]), "std": float(summary.loc[c, "std"])} for c in cols}},
          open(os.path.join(OUT_DIR, f"summary_{TAG}.json"), "w"), indent=2, ensure_ascii=False)

h = pd.DataFrame(histories)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for s, g in h.groupby("seed"):
    ax[0].plot(g["epoch"], g["val_nll"], marker="o", label=f"seed {s}")
    ax[1].plot(g["epoch"], g["val_macro_f1"], marker="o", label=f"seed {s}")
ax[0].set_title("Val NLL theo epoch (tăng dần = quá khớp)"); ax[1].set_title("Val Macro-F1 theo epoch")
for a in ax: a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f"curves_{TAG}.png"), dpi=150); plt.show()

## 9. Đo tốc độ suy luận (điền vào bảng so sánh)

| Phép đo | Ý nghĩa |
|---|---|
| GPU, padding cố định vs động | Thời gian chỉ riêng mô hình; padding động gần với thực tế hơn |
| End-to-end, batch = 1 | Từ văn bản đã làm sạch → tách từ → tokenizer → mô hình → nhãn. Đây là độ trễ thật mỗi bình luận khi chạy trong cascade |
| CPU | Nếu máy chủ triển khai không có GPU |

Mỗi phép đo có warm-up và lấy trung bình trên nhiều mẫu; GPU có `torch.cuda.synchronize()` trước khi bấm giờ. So sánh giữa các mô hình chỉ có nghĩa khi đo trong cùng một phiên (cùng GPU/CPU).

In [ ]:
import platform
ck = torch.load(os.path.join(OUT_DIR, f"{TAG}_best.pt"), map_location="cpu", weights_only=False)
m = build_model(); m.load_state_dict(ck["model_state_dict"]); m.to(DEVICE).eval()
if use_amp: torch.cuda.reset_peak_memory_stats()
def sync(dev):
    if dev.type == "cuda": torch.cuda.synchronize()

@torch.no_grad()
def time_b1(model, dev, items, dynamic, amp):
    lat = []
    for k, b in enumerate(items):
        x = collate([b], dynamic=dynamic); ids, am = x["input_ids"].to(dev), x["attention_mask"].to(dev)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):
            sync(dev); t = time.perf_counter(); model(ids, am); sync(dev)
        if k >= 10: lat.append((time.perf_counter() - t) * 1000)      # 10 mẫu đầu là warm-up
    return float(np.mean(lat)), float(np.percentile(lat, 95))

@torch.no_grad()
def time_throughput(model, dev, ds, dynamic, amp, bs=64):
    dl = DataLoader(ds, batch_size=bs, collate_fn=lambda b: collate(b, dynamic=dynamic)); n = 0
    for k, b in enumerate(dl):
        if k == 1: sync(dev); t = time.perf_counter(); n = 0            # batch đầu là warm-up
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):
            model(b["input_ids"].to(dev), b["attention_mask"].to(dev))
        n += len(b["labels"])
    sync(dev); return n / (time.perf_counter() - t)

@torch.no_grad()
def time_end_to_end(model, dev, texts, quotes, amp):
    seg_ms, tok_ms, mdl_ms = [], [], []
    for k, (t_, q_) in enumerate(zip(texts, quotes)):
        a = time.perf_counter(); ts, qs = segment_fn(t_), (segment_fn(q_) if INPUT_MODE == "text_quote" else "")
        b_ = time.perf_counter(); x = collate([{"ids": encode(ts, qs), "y": 0}], dynamic=True)
        ids, am = x["input_ids"].to(dev), x["attention_mask"].to(dev)
        c_ = time.perf_counter()
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):
            sync(dev); model(ids, am).argmax(-1).item(); sync(dev)
        d_ = time.perf_counter()
        if k >= 10: seg_ms.append((b_ - a) * 1000); tok_ms.append((c_ - b_) * 1000); mdl_ms.append((d_ - c_) * 1000)
    tot = np.array(seg_ms) + np.array(tok_ms) + np.array(mdl_ms)
    return {"segment_ms": float(np.mean(seg_ms)), "tokenize_ms": float(np.mean(tok_ms)), "model_ms": float(np.mean(mdl_ms)),
            "total_ms": float(tot.mean()), "total_p95_ms": float(np.percentile(tot, 95))}

N_B1, N_E2E, N_CPU = 310, 210, 110
items = [test_ds[i] for i in range(N_B1)]
raw_t, raw_q = test_df["text"].tolist(), test_df["quoted_text"].tolist()
speed = {"tag": TAG, "backbone": BACKBONE, "model_name": MODEL_NAME, "head": HEAD, "needs_seg": NEEDS_SEG,
         "input_mode": INPUT_MODE, "max_len": MAX_LEN,
         "gpu": torch.cuda.get_device_name(0) if use_amp else None,
         "cpu": f"{platform.processor() or platform.machine()} x{os.cpu_count()} threads",
         "params_million": sum(p.numel() for p in m.parameters()) / 1e6,
         "ckpt_mb": os.path.getsize(os.path.join(OUT_DIR, f"{TAG}_best.pt")) / 2**20}
if use_amp:
    for dyn in (False, True):
        k_ = "dynamic" if dyn else "fixed"
        speed[f"gpu_b1_{k_}_ms"], speed[f"gpu_b1_{k_}_p95_ms"] = time_b1(m, DEVICE, items, dyn, True)
        speed[f"gpu_b64_{k_}_per_s"] = time_throughput(m, DEVICE, test_ds, dyn, True)
    speed.update({f"gpu_e2e_{k}": v for k, v in time_end_to_end(m, DEVICE, raw_t[:N_E2E], raw_q[:N_E2E], True).items()})
    speed["gpu_peak_vram_infer_mb"] = torch.cuda.max_memory_allocated() / 2**20
    speed["gpu_peak_vram_train_mb"] = float(pd.DataFrame(results)["peak_vram_train_mb"].max())
    speed["train_samples_per_s"] = float(pd.DataFrame(results)["train_samples_per_s"].mean())
if MEASURE_CPU:
    cpu = torch.device("cpu"); torch.set_num_threads(os.cpu_count())
    mc = m.float().to(cpu).eval()
    speed["cpu_b1_dynamic_ms"], speed["cpu_b1_dynamic_p95_ms"] = time_b1(mc, cpu, items[:N_CPU], True, False)
    speed.update({f"cpu_e2e_{k}": v for k, v in time_end_to_end(mc, cpu, raw_t[:N_CPU], raw_q[:N_CPU], False).items()})
print(json.dumps({k: (round(v, 2) if isinstance(v, float) else v) for k, v in speed.items()}, indent=2, ensure_ascii=False))
json.dump(speed, open(os.path.join(OUT_DIR, f"speed_{TAG}.json"), "w"), indent=2, ensure_ascii=False)
del m; gc.collect(); torch.cuda.empty_cache()

## 10. So sánh các giai đoạn đã chạy
Đọc mọi `results_*.csv` trong `OUT_DIR` (hoặc thêm thư mục chứa kết quả các lần chạy trước vào `EXTRA_DIRS`).

In [ ]:
import glob
EXTRA_DIRS = []    # ví dụ ["/kaggle/input/ket-qua-stage0"]
files = sorted({f for d in [OUT_DIR] + EXTRA_DIRS for f in glob.glob(os.path.join(d, "results_*.csv"))})
if files:
    allr = pd.concat([pd.read_csv(f) for f in files])
    show = ["test_macro_f1", "f1_offensive", "f1_hate", "macro_f1_has_quote", "ece_raw", "ece_cal", "coverage_tier2"]
    g = allr.groupby("tag")[show]
    table = g.mean().round(4).astype(str) + " ± " + g.std(ddof=1).round(4).astype(str)
    table.insert(0, "n_seed", allr.groupby("tag").size())
    sp_files = sorted({f for d in [OUT_DIR] + EXTRA_DIRS for f in glob.glob(os.path.join(d, "speed_*.json"))})
    if sp_files:
        sp = pd.DataFrame([json.load(open(f)) for f in sp_files]).set_index("tag")
        keep = [c for c in ["params_million", "gpu_b1_dynamic_ms", "gpu_e2e_total_ms", "gpu_b64_dynamic_per_s", "cpu_e2e_total_ms",
                            "train_samples_per_s", "gpu_peak_vram_train_mb"] if c in sp.columns]
        table = table.join(sp[keep].round(1), how="left")
    display(table); table.to_csv(os.path.join(OUT_DIR, "compare_stages.csv"), encoding="utf-8-sig")

## Ghi chú khi viết báo cáo
- **Cách chạy**: lần lượt `STAGE = "0_baseline"`, `"1_recipe"`, `"2_quote"`. So sánh giai đoạn 1 với 2 rồi điền `STAGE3_INPUT_MODE`. Giai đoạn 3 chỉ chạy được sau khi nhóm gán lại nhãn và có `label_fixes.csv` (từ `export_fixes.py`).
- **Ablation từng thành phần** của giai đoạn 1 (nếu còn quota GPU): `OVERRIDES = {"rdrop_alpha": 0.0}`, `{"label_smoothing": 0.0}`, `{"llrd": 1.0}`.
- **Tập test không bao giờ bị sửa nhãn**, nên mọi giai đoạn so trên cùng một tập test (fingerprint test ở mục 1 phải giống nhau).
- **Coverage** là tỉ lệ bình luận trên test mà tầng 2 tự quyết với ngưỡng τ2 chọn trên val ở độ chính xác `TARGET_ACC`. Coverage cao hơn ở cùng độ chính xác nghĩa là ít phải gọi Qwen hơn.
- **So sánh mô hình**: giữ nguyên `STAGE` tốt nhất, lần lượt đổi `BACKBONE` = `phobert_cls`, `visobert_cls`, `visobert_cnn`. Với `phobert_cls` so với `phobert_cnn` là đóng góp của lớp CNN; `visobert_*` so với `phobert_*` là đóng góp của bộ mã hoá.
- **Tốc độ**: báo cáo `gpu_e2e_total_ms` (độ trễ thật mỗi bình luận) và `cpu_e2e_total_ms` nếu triển khai không có GPU. `gpu_e2e_segment_ms` cho thấy chi phí tách từ mà ViSoBERT không phải trả.
- Checkpoint `*_best.pt` và `*_calibration.json` dùng với `tier2_classifier.py` (`Tier2Classifier(ckpt, ...).load_calibration(json)`), module này tự biết backbone và có cần tách từ không.

## Gói kết quả để tải về
In danh sách file đã ghi và nén các file nhỏ (csv, json, png, npz) thành `results_bundle.zip`. Checkpoint (`*_best.pt` hoặc thư mục `*_best`) rất nặng nên để riêng.

In [ ]:
import zipfile, glob
print("Thư mục kết quả:", OUT_DIR)
rows = []
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*"))):
    if os.path.isfile(f): rows.append((os.path.basename(f), os.path.getsize(f) / 2**20))
    elif os.path.isdir(f) and f.endswith("_best"): rows.append((os.path.basename(f) + "/", sum(os.path.getsize(os.path.join(f, x)) for x in os.listdir(f)) / 2**20))
print(pd.DataFrame(rows, columns=["file", "MB"]).round(2).to_string(index=False))
small = [f for f in glob.glob(os.path.join(OUT_DIR, "*")) if os.path.isfile(f) and f.endswith((".csv", ".json", ".png", ".npz")) and "seg_cache" not in f]
ZIP_PATH = os.path.join(OUT_DIR, "results_bundle.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in small: z.write(f, os.path.basename(f))
print(f"\\nĐã nén {len(small)} file vào {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/2**20:.1f} MB)")
try:
    from google.colab import files; files.download(ZIP_PATH)      # chỉ chạy trên Colab
except ImportError:
    pass